In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
CATALOG = "workspace"
SCHEMA = "final"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_sales_order_detail"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_sales_order_detail"

In [0]:
detail_df = spark.table(BRONZE_TABLE)

display(detail_df)

In [0]:
print(
    f"Bronze Count : {detail_df.count()}"
)

In [0]:
def clean_text(column_name):

    cleaned = trim(
        regexp_replace(
            regexp_replace(
                col(column_name),
                "\\+",
                ""
            ),
            "&",
            ""
        )
    )

    return when(
        cleaned == "",
        None
    ).otherwise(cleaned)

In [0]:
null_analysis_df = detail_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in detail_df.columns
])

display(null_analysis_df)

In [0]:
detail_clean_df = (
    detail_df

    .withColumn(
        "CarrierTrackingNumber",
        clean_text("CarrierTrackingNumber")
    )

    .withColumn(
        "rowguid",
        clean_text("rowguid")
    )
)

In [0]:
detail_clean_df = (
    detail_clean_df

    .withColumn(
        "SalesOrderID",
        expr(
            "try_cast(SalesOrderID as BIGINT)"
        )
    )

    .withColumn(
        "SalesOrderDetailID",
        expr(
            "try_cast(SalesOrderDetailID as BIGINT)"
        )
    )

    .withColumn(
        "OrderQty",
        expr(
            "try_cast(OrderQty as INT)"
        )
    )

    .withColumn(
        "ProductID",
        expr(
            "try_cast(ProductID as BIGINT)"
        )
    )

    .withColumn(
        "SpecialOfferID",
        expr(
            "try_cast(SpecialOfferID as INT)"
        )
    )

    .withColumn(
        "UnitPrice",
        expr(
            "try_cast(UnitPrice as DOUBLE)"
        )
    )

    .withColumn(
        "UnitPriceDiscount",
        expr(
            "try_cast(UnitPriceDiscount as DOUBLE)"
        )
    )

    .withColumn(
        "LineTotal",
        expr(
            "try_cast(LineTotal as DOUBLE)"
        )
    )
)

In [0]:
detail_clean_df = (
    detail_clean_df
    .withColumn(
        "ModifiedDate",
        expr(
            "try_cast(ModifiedDate as timestamp)"
        )
    )
)

In [0]:
detail_valid_df = (
    detail_clean_df

    .filter(
        col("SalesOrderID").isNotNull()
    )

    .filter(
        col("ProductID").isNotNull()
    )

    .filter(
        col("OrderQty") > 0
    )

    .filter(
        col("UnitPrice") >= 0
    )
)

In [0]:
# Gross Amount
detail_valid_df = (
    detail_valid_df
    .withColumn(
        "CalculatedLineAmount",
        round(
            col("OrderQty") *
            col("UnitPrice"),
            2
        )
    )
)

# Discount Amount
detail_valid_df = (
    detail_valid_df
    .withColumn(
        "DiscountAmount",
        round(
            col("CalculatedLineAmount")
            *
            col("UnitPriceDiscount"),
            2
        )
    )
)

# Net Sales Amount
detail_valid_df = (
    detail_valid_df
    .withColumn(
        "NetSalesAmount",
        round(
            col("CalculatedLineAmount")
            -
            col("DiscountAmount"),
            2
        )
    )
)

In [0]:
window_spec = (
    Window
    .partitionBy(
        "SalesOrderID",
        "SalesOrderDetailID"
    )
    .orderBy(
        col("ingested_at").desc()
    )
)

detail_dedup_df = (
    detail_valid_df
    .withColumn(
        "rn",
        row_number().over(window_spec)
    )
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
dq_order = detail_dedup_df.filter(
    col("SalesOrderID").isNull()
).count()

dq_product = detail_dedup_df.filter(
    col("ProductID").isNull()
).count()

dq_qty = detail_dedup_df.filter(
    col("OrderQty") <= 0
).count()

dq_price = detail_dedup_df.filter(
    col("UnitPrice") < 0
).count()

dq_dup = (
    detail_dedup_df
    .groupBy(
        "SalesOrderID",
        "SalesOrderDetailID"
    )
    .count()
    .filter(col("count") > 1)
    .count()
)

print(f"Null SalesOrderID      : {dq_order}")
print(f"Null ProductID         : {dq_product}")
print(f"Invalid OrderQty       : {dq_qty}")
print(f"Negative UnitPrice     : {dq_price}")
print(f"Duplicate Business Key : {dq_dup}")

In [0]:
detail_dedup_df = (
    detail_dedup_df
    .withColumn(
        "sales_detail_hash",
        sha2(
            concat_ws(
                "|",
                col("ProductID").cast("string"),
                col("OrderQty").cast("string"),
                col("UnitPrice").cast("string"),
                col("UnitPriceDiscount").cast("string"),
                col("LineTotal").cast("string")
            ),
            256
        )
    )
)

In [0]:
silver_detail_df = (
    detail_dedup_df
    .withColumn(
        "silver_load_timestamp",
        current_timestamp()
    )
    .withColumn(
        "silver_layer",
        lit("SILVER")
    )
)

In [0]:
(
    silver_detail_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(SILVER_TABLE)
)

In [0]:
silver_df = spark.table(SILVER_TABLE)

print(
    f"Silver Count : {silver_df.count()}"
)

display(silver_df)